### Fluorine-18 Decay and PET Scan Timing

Fluorine-18 (${}^{18}\text{F}$) is a positron-emitting radioisotope with a
half-life of $t_{1/2} = 1.833$ hours. It is attached to glucose molecules to
form **FDG** (fluorodeoxyglucose), which is injected into patients before a
**PET scan** (Positron Emission Tomography). Cancer cells absorb FDG at a higher
rate than normal tissue, making them visible as bright spots in the scan.

Because ${}^{18}\text{F}$ decays rapidly, the scan must be completed while
enough radioactive signal remains. That deadline is what makes this isotope
worth simulating. The clock in `carbon14_decay.ipynb` runs for tens of thousands
of years; this one runs out during the patient's visit, so the same equation now
has to answer a scheduling question.

As derived in the Carbon-14 notebook, the decay equation is:

$$\frac{dn}{dt} = -\frac{\ln 2}{t_{1/2}}\,n
\qquad\Rightarrow\qquad
n(t) = n_0\,2^{-t/t_{1/2}}$$

This notebook uses Euler's method to simulate the decay, then uses the analytic
solution to compute the hard time limit for a useful PET scan.

---
### Setup: parameters and initial conditions

The simulation runs for 12 hours (about 6.5 half-lives) with 100 steps of
$\Delta t = 0.12$ hours (7.2 minutes each). The half-life of ${}^{18}\text{F}$
is $t_{1/2} = 1.833$ hours (109.97 minutes), and the simulation starts at 100%
concentration.

Everything is allocated and initialized in this one cell. The Carbon-14 notebook
spread the same work over three cells to introduce the pieces one at a time;
here the model is already familiar, so it is set up in a single step.

Watch the step size relative to the half-life. Carbon-14 used $\Delta t$ of ten
years against a 5,730-year half-life, so each step advanced almost nothing. Here
each step covers about 6.5% of a half-life, which is coarse enough that Euler's
error will be visible in the plot at the end.

In [ ]:
"""fluorine18_decay.ipynb"""

# Cell 01 - Simulation parameters

%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

tf = 12  # final time (hours)
ts = 100  # number of time steps
dt = tf / ts  # time step size (0.12 hours = 7.2 minutes)

t = np.zeros(ts)  # Array for time (hours)
n = np.zeros(ts)  # Array for concentration (percentage)

n[0] = 100  # initial concentration (100%)
tau = 1.833  # half-life of F-18 (hours)

print(f"{tf=:,}  {ts=:,}  {dt=:.2f}")

---
### Euler forward integration

The decay constant $\lambda = \ln 2 / t_{1/2}$ converts the half-life into the
correct rate for the ODE $dn/dt = -\lambda\,n$.

The loop itself is identical to the one in the Carbon-14 notebook. Only the
half-life and the unit of time have changed, which is the point worth noticing:
one integrator handles both an archaeological clock and a clinical one.

In [ ]:
# Cell 02 - Euler forward integration of the F-18 decay equation

decay_const = np.log(2) / tau  # lambda = ln(2) / half-life

for i in range(ts - 1):
    t[i + 1] = t[i] + dt
    n[i + 1] = n[i] - decay_const * n[i] * dt

pd.DataFrame({"t": t[:5], "n": n[:5]})

---
### The decay curve and the PET scan time limit

As of 2026, the best PET detector technology can detect ${}^{18}\text{F}$
concentrations down to approximately **15%** of the original dose.
Below this level the signal-to-noise ratio is too poor to produce
a diagnostic image.

Solving the analytic solution for the time at which 15% remains:

$$
0.15 = 2^{-t_{\max}/t_{1/2}}
\qquad\Rightarrow\qquad
t_{\max}
=
-t_{1/2}\,\log_2(0.15)
=
\frac{t_{1/2}\,\ln(1/0.15)}{\ln 2}
=
\frac{t_{1/2}\,\ln(6.667)}{\ln 2}
$$

Since

$$
\log_2\!\left(\frac{1}{0.15}\right)
=
\log_2(6.667)
\approx 2.74,
$$

the maximum useful imaging time is approximately

$$
t_{\max}
\approx
2.74\,t_{1/2}.
$$

For ${}^{18}\text{F}$, whose half-life is approximately 110 minutes,

$$
t_{\max}
\approx
2.74 \times 110
\approx
301\ \text{minutes}
\approx
5.0\ \text{hours}.
$$

This gives the hard outer limit for any useful scan.
In clinical practice, scans are performed well before this limit
(typically 60 to 90 minutes post-injection) to ensure sufficient image quality.

**A word on the gap between the two curves.** Unlike the Carbon-14 plot, the
solid Euler trace here sits visibly below the dashed analytic one, by up to
about 0.85 percentage points near $t = 2.6$ hours. This is the coarse step size
showing itself, exactly as expected: Euler follows the tangent line, the tangent
line lies below a concave-up curve, and so the numerical solution decays a
little too fast. The relative error grows steadily, reaching about 10% by the
end of the 12-hour run, which is well past the point where the isotope is
clinically useful anyway.

In [ ]:
# Cell 03 - Plot decay curve and compute maximum scan window

n_min_pct = 15.0  # detection limit (%)
t_max = -tau * np.log(n_min_pct / 100) / np.log(2)  # hours

print(f"Detection limit      : {n_min_pct}% of original F-18")
print(f"Half-lives elapsed   : {t_max / tau:.2f}")
print(f"Maximum scan window  : {t_max:.2f} hours ({t_max * 60:.0f} minutes)")

t_exact = np.linspace(0, tf, 500)
n_exact = 100 * 2 ** (-t_exact / tau)

plt.figure("fluorine18_decay", figsize=(9, 5))
plt.plot(t, n, label="Euler's method")
plt.plot(t_exact, n_exact, "--", label=r"Analytic $n_0\,2^{-t/\tau}$")
plt.axhline(
    n_min_pct, color="red", linestyle=":", label=f"Detection limit ({n_min_pct}%)"
)
plt.axvline(
    t_max,
    color="orange",
    linestyle=":",
    label=f"Max scan window ({t_max:.2f} hr / {t_max * 60:.0f} min)",
)
plt.scatter([t_max], [n_min_pct], color="red", zorder=5)
plt.title("Fluorine-18 Decay - PET Scan Timing")
plt.xlabel("Time (hours)")
plt.ylabel("% Concentration")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()